# Name That LLM: Starter Baseline (TF-IDF + Logistic Regression)


In [2]:
import pathlib

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline

CANDIDATES = [
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('/kaggle/input/name-that-llm'),
]
DATA_DIR = next(p for p in CANDIDATES if (p / 'train.csv').exists())

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train_df.shape} | Unique models: {train_df["model"].nunique()} | Prompts: {train_df["prompt_id"].nunique()}')
print(f'Test shape: {test_df.shape}')
train_df.head(3)

Train shape: (2699, 4) | Unique models: 18 | Prompts: 150
Test shape: (3, 2)


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash


In [7]:
# Build Word + Character N-Gram TF-IDF + Logistic Regression Pipeline
model_pipeline = Pipeline(
    [
        (
            'features',
            FeatureUnion(
                [
                    (
                        'word_tfidf',
                        TfidfVectorizer(
                            ngram_range=(1, 2),
                            max_features=25000,
                            sublinear_tf=True,
                        ),
                    ),
                    (
                        'char_tfidf',
                        TfidfVectorizer(
                            analyzer='char',
                            ngram_range=(3, 5),
                            max_features=35000,
                            sublinear_tf=True,
                        ),
                    ),
                ]
            ),
        ),
        ('clf', LinearSVC(C=9)),
    ]
)

# 5-fold CV that keeps each prompt in a single fold (test prompts are unseen), scored like the leaderboard
cv_scores = cross_val_score(
    model_pipeline,
    train_df['text'].fillna(''),
    train_df['model'],
    groups=train_df['prompt_id'],
    cv=GroupKFold(n_splits=5),
    scoring='balanced_accuracy',
    n_jobs=-1,
)
print(f'5-Fold Grouped CV Balanced Accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Fit on full training set
model_pipeline.fit(train_df['text'].fillna(''), train_df['model'])

5-Fold Grouped CV Balanced Accuracy: 0.2960 +/- 0.0141


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=25000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True)),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char',
                                                                 max_features=35000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 sublinear_tf=True))])),
                ('clf', LinearSVC(C=9))])

In [6]:
# Batch inference on test.csv (1 example row interactively; the full hidden test set during the Leaderboard Rerun)
test_preds = model_pipeline.predict(test_df['text'].fillna(''))

submission = pd.DataFrame(
    {
        'id': test_df['id'],
        'model': test_preds,
    }
)

submission.to_parquet('submission.parquet', index=False)
print(f'Saved submission.parquet with {len(submission)} rows.')
submission.head(10)

Saved submission.parquet with 3 rows.


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-sol
2,0062a53d5487,gemini-3.7-flash


In [14]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Prepare data
X = train_df['text'].fillna('')
y = train_df['model']
groups = train_df['prompt_id']

# Same validation strategy as the original notebook
cv = GroupKFold(n_splits=5)

results = []

# Experiment with different C values
for c in [0.1, 1, 3, 9, 20]:

    model_pipeline = Pipeline([
        ('features', FeatureUnion([
            ('word_tfidf', TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ('char_tfidf', TfidfVectorizer(
                analyzer='char',
                ngram_range=(3, 5),
                max_features=35000,
                sublinear_tf=True,
            )),
        ])),
        ('clf', LinearSVC(C=c)),
    ])

    # Calculate scores using the same method as the original notebook
    cv_scores = cross_val_score(
        model_pipeline,
        X,
        y,
        groups=groups,
        cv=cv,
        scoring='balanced_accuracy',
        n_jobs=-1,
    )

    results.append({
        'C': c,
        'Mean balanced accuracy': cv_scores.mean(),
        'Std deviation': cv_scores.std(),
    })

    print(
        f'C={c}: {cv_scores.mean():.4f} '
        f'+/- {cv_scores.std():.4f}'
    )

# Display results from best to worst
results_df = pd.DataFrame(results).sort_values(
    'Mean balanced accuracy',
    ascending=False,
).reset_index(drop=True)

display(results_df)

# Select the best C value
best_c = results_df.loc[0, 'C']
print(f'Best C: {best_c}')

# Train the selected pipeline on all training data
model_pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char',
            ngram_range=(3, 5),
            max_features=35000,
            sublinear_tf=True,
        )),
    ])),
    ('clf', LinearSVC(C=best_c)),
])

model_pipeline.fit(X, y)

print(f'Final model trained using C={best_c}')

C=0.1: 0.3045 +/- 0.0123
C=1: 0.2989 +/- 0.0119
C=3: 0.2986 +/- 0.0126
C=9: 0.2960 +/- 0.0141
C=20: 0.2978 +/- 0.0142


,C,Mean balanced accuracy,Std deviation
0,0.1,0.304470,0.012335
1,1.0,0.298927,0.011885
2,3.0,0.298557,0.012609
3,20.0,0.297816,0.014197
4,9.0,0.295964,0.014149


Best C: 0.1
Final model trained using C=0.1


In [15]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


# 1. Prepare data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# Use exactly the same five grouped folds for every experiment
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# 2. Define the feature configurations
configs = [
    {
        "name": "Baseline",
        "word_ngram": (1, 2),
        "char_analyzer": "char",
        "char_ngram": (3, 5),
    },
    {
        "name": "Word ngrams (1, 3)",
        "word_ngram": (1, 3),
        "char_analyzer": "char",
        "char_ngram": (3, 5),
    },
    {
        "name": "Character ngrams (2, 6)",
        "word_ngram": (1, 2),
        "char_analyzer": "char",
        "char_ngram": (2, 6),
    },
    {
        "name": "Character within words",
        "word_ngram": (1, 2),
        "char_analyzer": "char_wb",
        "char_ngram": (3, 5),
    },
]


# 3. Build the pipeline
def build_pipeline(config):
    return Pipeline([
        (
            "features",
            FeatureUnion([
                (
                    "word_tfidf",
                    TfidfVectorizer(
                        ngram_range=config["word_ngram"],
                        max_features=25000,
                        sublinear_tf=True,
                    ),
                ),
                (
                    "char_tfidf",
                    TfidfVectorizer(
                        analyzer=config["char_analyzer"],
                        ngram_range=config["char_ngram"],
                        max_features=35000,
                        sublinear_tf=True,
                    ),
                ),
            ]),
        ),
        ("clf", LinearSVC(C=0.1)),
    ])


# 4. Evaluate each configuration
results = []

for config in configs:
    print(f"\nEvaluating: {config['name']}")

    experiment = build_pipeline(config)

    scores = cross_val_score(
        experiment,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Experiment": config["name"],
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": [round(float(s), 4) for s in scores],
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")


# 5. Compare results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== EXPERIMENT COMPARISON ==========")
display(results_df)

best_name = results_df.loc[0, "Experiment"]
best_config = next(
    config for config in configs
    if config["name"] == best_name
)

print(f"Best configuration: {best_name}")
print(
    f"Best mean balanced accuracy: "
    f"{results_df.loc[0, 'Mean balanced accuracy']:.4f}"
)


# 6. Train the selected model on all training data
model_pipeline = build_pipeline(best_config)
model_pipeline.fit(X, y)

print("Final model trained.")





Evaluating: Baseline
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123

Evaluating: Word ngrams (1, 3)
Fold scores: [0.287  0.2759 0.3185 0.3037 0.3094]
Mean: 0.2989 +/- 0.0154

Evaluating: Character ngrams (2, 6)
Fold scores: [0.2889 0.2741 0.3093 0.313  0.3132]
Mean: 0.2997 +/- 0.0156

Evaluating: Character within words
Fold scores: [0.2815 0.2907 0.2907 0.3019 0.289 ]
Mean: 0.2908 +/- 0.0065

========== EXPERIMENT COMPARISON ==========


,Experiment,Mean balanced accuracy,Std deviation,Fold scores
0,Baseline,0.304470,0.012335,"[0.2926, 0.287, 0.3111, 0.3185, 0.3131]"
1,"Character ngrams (2, 6)",0.299668,0.015632,"[0.2889, 0.2741, 0.3093, 0.313, 0.3132]"
2,"Word ngrams (1, 3)",0.298914,0.015404,"[0.287, 0.2759, 0.3185, 0.3037, 0.3094]"
3,Character within words,0.290766,0.006514,"[0.2815, 0.2907, 0.2907, 0.3019, 0.289]"


Best configuration: Baseline
Best mean balanced accuracy: 0.3045
Final model trained.


In [16]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# Prepare data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# Keep exactly the same five grouped validation folds
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# Build the same baseline pipeline
def build_pipeline(class_weight=None):
    return Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char_tfidf", TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
                max_features=35000,
                sublinear_tf=True,
            )),
        ])),
        ("clf", LinearSVC(
            C=0.1,
            class_weight=class_weight,
        )),
    ])


# Compare normal training vs balanced class weights
results = []

for weight in [None, "balanced"]:
    print(f"\nTesting class_weight={weight}")

    model = build_pipeline(class_weight=weight)

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "class_weight": str(weight),
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")


# Compare the results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Testing class_weight=None
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123

Testing class_weight=balanced
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123


,class_weight,Mean balanced accuracy,Std deviation
0,None,0.30447,0.012335
1,balanced,0.30447,0.012335


In [17]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# 1. Prepare the data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# 2. Keep the same validation folds for every experiment
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Check class distribution
print("Training examples per LLM:")
display(train_df["model"].value_counts().sort_values())

# 3. Define experiments
configs = [
    {"name": "Baseline", "word": 1.0, "char": 1.0},
    {"name": "Word lighter", "word": 0.5, "char": 1.0},
    {"name": "Word stronger", "word": 2.0, "char": 1.0},
    {"name": "Character lighter", "word": 1.0, "char": 0.5},
    {"name": "Character stronger", "word": 1.0, "char": 2.0},
]

# 4. Build the model
def build_pipeline(config):
    return Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word_tfidf", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char_tfidf", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(3, 5),
                    max_features=35000,
                    sublinear_tf=True,
                )),
            ],
            transformer_weights={
                "word_tfidf": config["word"],
                "char_tfidf": config["char"],
            },
        )),
        ("clf", LinearSVC(C=0.1)),
    ])

# 5. Evaluate each configuration
results = []

for config in configs:
    print(f"\nTesting: {config['name']}")

    model = build_pipeline(config)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "Experiment": config["name"],
        "Word weight": config["word"],
        "Character weight": config["char"],
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

# 6. Compare results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

# 7. Select and train the best configuration
best_name = results_df.loc[0, "Experiment"]
best_config = next(
    c for c in configs if c["name"] == best_name
)

model_pipeline = build_pipeline(best_config)
model_pipeline.fit(X, y)

print(f"Best configuration: {best_name}")
print(
    "Best balanced accuracy:",
    round(results_df.loc[0, "Mean balanced accuracy"], 4),
)

Training examples per LLM:


model
claude-fable-5            149
qwen3.8-max               150
gemini-3.6-flash          150
claude-sonnet-5           150
glm-5.3-flash             150
gemini-3.1-pro-preview    150
gemini-3.8-flash          150
gpt-5.6-terra             150
gpt-5.6-luna              150
claude-opus-4-8           150
gemini-3.7-flash          150
claude-opus-5             150
kimi-k3                   150
gpt-5.5                   150
glm-5.3                   150
claude-opus-4-6           150
gpt-6-astra               150
gpt-5.6-sol               150
Name: count, dtype: int64


Testing: Baseline
Fold scores: [0.2926 0.287  0.3111 0.3185 0.3131]
Mean: 0.3045 +/- 0.0123

Testing: Word lighter
Fold scores: [0.3019 0.2611 0.3056 0.3185 0.3131]
Mean: 0.3000 +/- 0.0203

Testing: Word stronger
Fold scores: [0.2685 0.2944 0.287  0.3037 0.2835]
Mean: 0.2874 +/- 0.0117

Testing: Character lighter
Fold scores: [0.2648 0.2907 0.2963 0.3019 0.2983]
Mean: 0.2904 +/- 0.0133

Testing: Character stronger
Fold scores: [0.3037 0.2778 0.3037 0.3315 0.315 ]
Mean: 0.3063 +/- 0.0175


,Experiment,Word weight,Character weight,Mean balanced accuracy,Std deviation
0,Character stronger,1.0,2.0,0.306335,0.017532
1,Baseline,1.0,1.0,0.304470,0.012335
2,Word lighter,0.5,1.0,0.300026,0.020302
3,Character lighter,1.0,0.5,0.290396,0.013287
4,Word stronger,2.0,1.0,0.287433,0.011727


Best configuration: Character stronger
Best balanced accuracy: 0.3063


In [18]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

# Prepare data
X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

# Same five-fold grouped validation
cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

# Try different regularization strengths
C_VALUES = [0.03, 0.1, 0.3, 1.0]
results = []

for c in C_VALUES:
    print(f"\nTesting C={c}")

    model = Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word_tfidf", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),
                ("char_tfidf", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(3, 5),
                    max_features=35000,
                    sublinear_tf=True,
                )),
            ],
            transformer_weights={
                "word_tfidf": 1.0,
                "char_tfidf": 2.0,
            },
        )),
        ("clf", LinearSVC(C=c)),
    ])

    scores = cross_val_score(
        model,
        X,
        y,
        groups=groups,
        cv=cv_splits,
        scoring="balanced_accuracy",
        n_jobs=-1,
    )

    results.append({
        "C": c,
        "Mean balanced accuracy": scores.mean(),
        "Std deviation": scores.std(),
        "Fold scores": scores.round(4).tolist(),
    })

    print(f"Fold scores: {scores.round(4)}")
    print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")

# Compare results
results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Testing C=0.03
Fold scores: [0.3056 0.2667 0.3074 0.3259 0.3112]
Mean: 0.3034 +/- 0.0197

Testing C=0.1
Fold scores: [0.3037 0.2778 0.3037 0.3315 0.315 ]
Mean: 0.3063 +/- 0.0175

Testing C=0.3
Fold scores: [0.2963 0.2852 0.2981 0.3315 0.3132]
Mean: 0.3049 +/- 0.0160

Testing C=1.0
Fold scores: [0.2907 0.287  0.2944 0.3259 0.3113]
Mean: 0.3019 +/- 0.0146


,C,Mean balanced accuracy,Std deviation,Fold scores
0,0.10,0.306335,0.017532,"[0.3037, 0.2778, 0.3037, 0.3315, 0.315]"
1,0.30,0.304853,0.016019,"[0.2963, 0.2852, 0.2981, 0.3315, 0.3132]"
2,0.03,0.303359,0.019692,"[0.3056, 0.2667, 0.3074, 0.3259, 0.3112]"
3,1.00,0.301890,0.014606,"[0.2907, 0.287, 0.2944, 0.3259, 0.3113]"


In [20]:
import re
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score


# ============================================================
# 1. PREPARE DATA AND VALIDATION FOLDS
# ============================================================

X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# ============================================================
# 2. CREATE A STYLISTIC FEATURE EXTRACTOR
# ============================================================

class StyleFeatureExtractor(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "as an ai",
        "as a language model",
        "certainly",
        "of course",
        "here is",
        "here's",
        "let's",
        "in conclusion",
        "in summary",
        "to summarize",
        "for example",
        "for instance",
        "note that",
        "it is important",
        "it's important",
        "overall",
        "however",
        "therefore",
        "moreover",
        "furthermore",
        "firstly",
        "secondly",
        "hope this helps",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract_features(text) for text in X],
            dtype=np.float32,
        )

    def extract_features(self, value):

        text = "" if value is None else str(value)
        lower = text.lower()

        char_count = max(len(text), 1)

        words = re.findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)

        word_lengths = np.asarray(
            [len(word) for word in words],
            dtype=np.float32,
        )

        sentences = [
            s.strip()
            for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = np.asarray(
            [
                len(re.findall(r"\b\w+\b", sentence))
                for sentence in sentences
            ],
            dtype=np.float32,
        )

        lines = text.splitlines() or [text]

        paragraphs = [
            p for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        # Basic length and vocabulary statistics
        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),

            float(word_lengths.mean()) if word_lengths.size else 0.0,
            float(word_lengths.std()) if word_lengths.size else 0.0,
            float(word_lengths.max()) if word_lengths.size else 0.0,

            len(set(words)) / word_count,

            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,

            float(sentence_lengths.mean())
            if sentence_lengths.size else 0.0,

            float(sentence_lengths.std())
            if sentence_lengths.size else 0.0,

            float(sentence_lengths.max())
            if sentence_lengths.size else 0.0,
        ]

        # Character composition
        features.extend([
            sum(c.isupper() for c in text) / char_count,
            sum(c.islower() for c in text) / char_count,
            sum(c.isdigit() for c in text) / char_count,
            sum(c.isspace() for c in text) / char_count,
            sum(ord(c) > 127 for c in text) / char_count,
        ])

        # Typography: frequency of punctuation per 1,000 characters
        for mark in self.punctuation:
            features.append(
                text.count(mark) / char_count * 1000
            )

        # Formatting and list indicators
        bullet_lines = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )

        features.extend([
            bullet_lines / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        # Common stylistic phrase indicators
        for phrase in self.phrases:
            features.append(float(phrase in lower))

        return features


# ============================================================
# 3. BUILD THREE SPECIALIZED CLASSIFIERS
# ============================================================

# Model A: word vocabulary and phrases
word_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=25000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1500,
    )),
])


# Model B: character patterns, including individual characters
char_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(1, 5),
        max_features=100000,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1500,
    )),
])


# Model C: writing style and typography
style_model = Pipeline([
    ("style_features", StyleFeatureExtractor()),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        C=1.0,
        max_iter=1500,
    )),
])


# ============================================================
# 4. GENERATE OUT-OF-FOLD PROBABILITIES
# ============================================================

# Each prediction is made by a model that did not train on
# that example or on other examples from the same prompt fold.

print("Evaluating word model...")
word_oof = cross_val_predict(
    word_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

print("Evaluating character model...")
char_oof = cross_val_predict(
    char_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

print("Evaluating style model...")
style_oof = cross_val_predict(
    style_model, X, y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)


# ============================================================
# 5. BLEND MODEL PROBABILITIES
# ============================================================

def geometric_blend(probabilities, weights):

    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        if weight > 0:
            log_prob += weight * np.log(
                np.clip(prob, 1e-12, 1.0)
            )

    # Normalize probabilities for each passage
    log_prob -= log_prob.max(axis=1, keepdims=True)

    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


oof_predictions = {
    "Word TF-IDF": word_oof,
    "Character TF-IDF": char_oof,
    "Style features": style_oof,

    "Word + character": geometric_blend(
        [word_oof, char_oof],
        [0.5, 0.5],
    ),

    "All three equally": geometric_blend(
        [word_oof, char_oof, style_oof],
        [1/3, 1/3, 1/3],
    ),

    "Style at 20%": geometric_blend(
        [word_oof, char_oof, style_oof],
        [0.4, 0.4, 0.2],
    ),
}


# ============================================================
# 6. COMPARE USING BALANCED ACCURACY
# ============================================================

results = []

for name, probabilities in oof_predictions.items():

    predictions = labels[np.argmax(probabilities, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "Method": name,
        "Balanced accuracy": score,
    })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

best_method = results_df.loc[0, "Method"]

print("Best method:", best_method)
print(
    "Best grouped-CV balanced accuracy:",
    round(results_df.loc[0, "Balanced accuracy"], 4),
)


# ============================================================
# 7. FIT ALL THREE MODELS ON FULL TRAINING DATA
# ============================================================

word_model.fit(X, y)
char_model.fit(X, y)
style_model.fit(X, y)


# ============================================================
# 8. PREDICT TEST DATA WITH THE BEST OOF METHOD
# ============================================================

X_test = test_df["text"].fillna("")

word_test = word_model.predict_proba(X_test)
char_test = char_model.predict_proba(X_test)
style_test = style_model.predict_proba(X_test)

test_predictions = {
    "Word TF-IDF": word_test,
    "Character TF-IDF": char_test,
    "Style features": style_test,

    "Word + character": geometric_blend(
        [word_test, char_test],
        [0.5, 0.5],
    ),

    "All three equally": geometric_blend(
        [word_test, char_test, style_test],
        [1/3, 1/3, 1/3],
    ),

    "Style at 20%": geometric_blend(
        [word_test, char_test, style_test],
        [0.4, 0.4, 0.2],
    ),
}

final_predictions = labels[
    np.argmax(test_predictions[best_method], axis=1)
]



Evaluating word model...
Evaluating character model...
Evaluating style model...


,Method,Balanced accuracy
0,Style at 20%,0.348548
1,All three equally,0.325593
2,Character TF-IDF,0.312617
3,Word + character,0.300758
4,Style features,0.285585
5,Word TF-IDF,0.262242


Best method: Style at 20%
Best grouped-CV balanced accuracy: 0.3485


In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score


# Combine probabilities using a weighted arithmetic mean
def arithmetic_blend(probabilities, weights):
    combined = sum(
        weight * prob
        for prob, weight in zip(probabilities, weights)
    )

    return combined / combined.sum(axis=1, keepdims=True)


# Combine probabilities using a weighted geometric mean
def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    # Numerical stability
    log_prob -= log_prob.max(axis=1, keepdims=True)

    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


# OOF probabilities generated using the same GroupKFold splits
oof_probabilities = [word_oof, char_oof, style_oof]

# Try a small, predefined set of weight combinations
experiments = [
    ("Geometric", (0.20, 0.60, 0.20)),
    ("Geometric", (0.30, 0.50, 0.20)),
    ("Geometric", (0.40, 0.40, 0.20)),
    ("Geometric", (0.50, 0.30, 0.20)),
    ("Geometric", (0.60, 0.20, 0.20)),
    ("Geometric", (0.45, 0.45, 0.10)),
    ("Geometric", (0.35, 0.35, 0.30)),
    ("Geometric", (0.30, 0.30, 0.40)),
    ("Geometric", (0.25, 0.25, 0.50)),
    ("Arithmetic", (0.40, 0.40, 0.20)),
    ("Arithmetic", (0.30, 0.50, 0.20)),
    ("Arithmetic", (0.50, 0.30, 0.20)),
    ("Arithmetic", (1/3, 1/3, 1/3)),
]

results = []

for blend_type, weights in experiments:

    if blend_type == "Geometric":
        probabilities = geometric_blend(
            oof_probabilities, weights
        )
    else:
        probabilities = arithmetic_blend(
            oof_probabilities, weights
        )

    predictions = labels[
        np.argmax(probabilities, axis=1)
    ]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "Blend": blend_type,
        "Word weight": weights[0],
        "Character weight": weights[1],
        "Style weight": weights[2],
        "Balanced accuracy": score,
    })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print(
    "Best blend score:",
    round(results_df.loc[0, "Balanced accuracy"], 4)
)

,Blend,Word weight,Character weight,Style weight,Balanced accuracy
0,Geometric,0.450000,0.450000,0.100000,0.357808
1,Geometric,0.200000,0.600000,0.200000,0.354479
2,Geometric,0.300000,0.500000,0.200000,0.351514
3,Geometric,0.400000,0.400000,0.200000,0.348548
4,Geometric,0.500000,0.300000,0.200000,0.344842
5,Geometric,0.600000,0.200000,0.200000,0.342622
6,Geometric,0.350000,0.350000,0.300000,0.328558
7,Geometric,0.300000,0.300000,0.400000,0.317808
8,Arithmetic,0.300000,0.500000,0.200000,0.317069
9,Arithmetic,0.500000,0.300000,0.200000,0.317069


Best blend score: 0.3578


In [22]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(probabilities[0], dtype=float)

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


# Reuse the existing out-of-fold probabilities
probabilities = [word_oof, char_oof, style_oof]

results = []

# Search combinations close to the current best
for style_weight in [0.05, 0.10, 0.15]:
    for word_weight in [0.35, 0.40, 0.45, 0.50, 0.55]:

        char_weight = 1.0 - word_weight - style_weight

        weights = [
            word_weight,
            char_weight,
            style_weight,
        ]

        blended = geometric_blend(probabilities, weights)

        predictions = labels[np.argmax(blended, axis=1)]

        score = balanced_accuracy_score(y, predictions)

        results.append({
            "Word weight": word_weight,
            "Character weight": char_weight,
            "Style weight": style_weight,
            "Balanced accuracy": score,
        })

results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df.head(10))

,Word weight,Character weight,Style weight,Balanced accuracy
0,0.40,0.50,0.10,0.358919
1,0.45,0.45,0.10,0.357808
2,0.35,0.55,0.10,0.357069
3,0.50,0.40,0.10,0.355958
4,0.35,0.50,0.15,0.355585
5,0.55,0.35,0.10,0.354104
6,0.35,0.60,0.05,0.354096
7,0.40,0.55,0.05,0.352988
8,0.40,0.45,0.15,0.351884
9,0.50,0.35,0.15,0.351879


In [23]:
import numpy as np
import pandas as pd

from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score


# --------------------------------------------------
# 1. Build a new model: TF-IDF + calibrated Linear SVM
# --------------------------------------------------

svm_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
    ])),
    ("clf", LinearSVC(C=0.1)),
])

# Calibration converts decision scores into estimated probabilities
try:
    calibrated_svm = CalibratedClassifierCV(
        estimator=svm_pipeline,
        method="sigmoid",
        cv=3,
    )
except TypeError:
    # Compatibility with older scikit-learn versions
    calibrated_svm = CalibratedClassifierCV(
        base_estimator=svm_pipeline,
        method="sigmoid",
        cv=3,
    )


# --------------------------------------------------
# 2. Generate grouped out-of-fold probabilities
# --------------------------------------------------

print("Evaluating calibrated SVM...")

svm_oof = cross_val_predict(
    calibrated_svm,
    X,
    y,
    cv=cv_splits,
    method="predict_proba",
    n_jobs=1,
)

svm_predictions = labels[np.argmax(svm_oof, axis=1)]

svm_score = balanced_accuracy_score(y, svm_predictions)

print(f"Calibrated SVM balanced accuracy: {svm_score:.4f}")


# --------------------------------------------------
# 3. Compare it with the current best blend
# --------------------------------------------------

def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


old_models = [word_oof, char_oof, style_oof]
old_weights = [0.40, 0.50, 0.10]

results = [{
    "Method": "Existing best blend",
    "SVM weight": 0.0,
    "Balanced accuracy": balanced_accuracy_score(
        y,
        labels[np.argmax(
            geometric_blend(old_models, old_weights),
            axis=1
        )]
    ),
}]

# Add the SVM at different weights
for svm_weight in [0.10, 0.20, 0.30, 0.40]:

    remaining = 1.0 - svm_weight

    weights = [
        old_weights[0] * remaining,
        old_weights[1] * remaining,
        old_weights[2] * remaining,
        svm_weight,
    ]

    probabilities = geometric_blend(
        [word_oof, char_oof, style_oof, svm_oof],
        weights,
    )

    predictions = labels[np.argmax(probabilities, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    results.append({
        "Method": "Blend with calibrated SVM",
        "SVM weight": svm_weight,
        "Balanced accuracy": score,
    })


results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)


Evaluating calibrated SVM...
Calibrated SVM balanced accuracy: 0.2696


,Method,SVM weight,Balanced accuracy
0,Existing best blend,0.0,0.358919
1,Blend with calibrated SVM,0.1,0.355578
2,Blend with calibrated SVM,0.2,0.349279
3,Blend with calibrated SVM,0.3,0.343346
4,Blend with calibrated SVM,0.4,0.330378


In [24]:
import numpy as np
import pandas as pd

from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import balanced_accuracy_score


# 1. Existing best ensemble
def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


base_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

base_predictions = labels[np.argmax(base_oof, axis=1)]
base_score = balanced_accuracy_score(y, base_predictions)

print(f"Existing ensemble: {base_score:.4f}")


# 2. Build Complement Naive Bayes using word + character TF-IDF
def build_nb_pipeline(alpha):
    return Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                ngram_range=(1, 2),
                max_features=25000,
                sublinear_tf=True,
            )),
            ("char_tfidf", TfidfVectorizer(
                analyzer="char",
                ngram_range=(3, 5),
                max_features=35000,
                sublinear_tf=True,
            )),
        ])),
        ("clf", ComplementNB(alpha=alpha)),
    ])


# 3. Evaluate different smoothing strengths
results = []

results.append({
    "Method": "Existing ensemble",
    "Alpha": None,
    "NB weight": 0.0,
    "Balanced accuracy": base_score,
})

for alpha in [0.01, 0.1, 0.5, 1.0]:

    print(f"\nEvaluating ComplementNB alpha={alpha}")

    nb_model = build_nb_pipeline(alpha)

    nb_oof = cross_val_predict(
        nb_model,
        X,
        y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=-1,
    )

    # Score ComplementNB alone
    nb_predictions = labels[np.argmax(nb_oof, axis=1)]
    nb_score = balanced_accuracy_score(y, nb_predictions)

    results.append({
        "Method": "ComplementNB alone",
        "Alpha": alpha,
        "NB weight": 1.0,
        "Balanced accuracy": nb_score,
    })

    print(f"ComplementNB score: {nb_score:.4f}")

    # Blend ComplementNB with the existing ensemble
    for nb_weight in [0.1, 0.2, 0.3]:

        blended = geometric_blend(
            [base_oof, nb_oof],
            [1.0 - nb_weight, nb_weight],
        )

        predictions = labels[np.argmax(blended, axis=1)]
        score = balanced_accuracy_score(y, predictions)

        results.append({
            "Method": "Ensemble + ComplementNB",
            "Alpha": alpha,
            "NB weight": nb_weight,
            "Balanced accuracy": score,
        })


# 4. Compare all experiments
results_df = pd.DataFrame(results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nBest experiment:")
display(results_df.head(5))

Existing ensemble: 0.3589

Evaluating ComplementNB alpha=0.01
ComplementNB score: 0.2337

Evaluating ComplementNB alpha=0.1
ComplementNB score: 0.2452

Evaluating ComplementNB alpha=0.5
ComplementNB score: 0.2570

Evaluating ComplementNB alpha=1.0
ComplementNB score: 0.2607


,Method,Alpha,NB weight,Balanced accuracy
0,Ensemble + ComplementNB,1.00,0.2,0.360027
1,Existing ensemble,NaN,0.0,0.358919
2,Ensemble + ComplementNB,0.10,0.1,0.357437
3,Ensemble + ComplementNB,1.00,0.1,0.357064
4,Ensemble + ComplementNB,0.50,0.1,0.356694
5,Ensemble + ComplementNB,0.01,0.1,0.356326
6,Ensemble + ComplementNB,0.50,0.2,0.356324
7,Ensemble + ComplementNB,0.01,0.2,0.354840
8,Ensemble + ComplementNB,0.10,0.2,0.354469
9,Ensemble + ComplementNB,1.00,0.3,0.354096



Best experiment:


,Method,Alpha,NB weight,Balanced accuracy
0,Ensemble + ComplementNB,1.0,0.2,0.360027
1,Existing ensemble,NaN,0.0,0.358919
2,Ensemble + ComplementNB,0.1,0.1,0.357437
3,Ensemble + ComplementNB,1.0,0.1,0.357064
4,Ensemble + ComplementNB,0.5,0.1,0.356694


In [25]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score


# =====================================================
# 1. MAP EACH MODEL TO ITS LAB
# =====================================================

MODEL_TO_FAMILY = {
    "gpt-5.5": "OpenAI",
    "gpt-5.6-luna": "OpenAI",
    "gpt-5.6-sol": "OpenAI",
    "gpt-5.6-terra": "OpenAI",
    "gpt-6-astra": "OpenAI",

    "gemini-3.1-pro-preview": "Google",
    "gemini-3.6-flash": "Google",
    "gemini-3.7-flash": "Google",
    "gemini-3.8-flash": "Google",

    "claude-fable-5": "Anthropic",
    "claude-opus-4-6": "Anthropic",
    "claude-opus-4-8": "Anthropic",
    "claude-opus-5": "Anthropic",
    "claude-sonnet-5": "Anthropic",

    "glm-5.3": "ZAI",
    "glm-5.3-flash": "ZAI",

    "qwen3.8-max": "Qwen",
    "kimi-k3": "Moonshot",
}


# =====================================================
# 2. PREPARE DATA
# =====================================================

X = train_df["text"].fillna("").reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

unknown_models = set(y.unique()) - set(MODEL_TO_FAMILY)

if unknown_models:
    raise ValueError(f"Missing family mappings: {unknown_models}")

all_models = np.array(sorted(y.unique()))
model_to_col = {
    model: i for i, model in enumerate(all_models)
}

families = sorted(set(MODEL_TO_FAMILY.values()))

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# =====================================================
# 3. CREATE TEXT FEATURES
# =====================================================

def build_features():
    return FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
    ])


# =====================================================
# 4. RUN TWO-STAGE GROUPED CROSS-VALIDATION
# =====================================================

oof_probs = np.zeros((len(X), len(all_models)))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv_splits, start=1):

    print(f"\n========== FOLD {fold}/5 ==========")

    X_train_text = X.iloc[train_idx]
    X_valid_text = X.iloc[valid_idx]

    y_train = y.iloc[train_idx].to_numpy()
    y_valid = y.iloc[valid_idx].to_numpy()

    # Learn features only from this fold's training texts
    features = build_features()

    X_train_features = features.fit_transform(X_train_text)
    X_valid_features = features.transform(X_valid_text)

    # Map training labels to their families
    y_train_family = np.array([
        MODEL_TO_FAMILY[model] for model in y_train
    ])

    # ---------------------------------------------
    # STAGE 1: PREDICT THE LAB
    # ---------------------------------------------

    family_model = LogisticRegression(
        C=1.0,
        max_iter=2000,
    )

    family_model.fit(
        X_train_features,
        y_train_family,
    )

    family_probs = family_model.predict_proba(
        X_valid_features
    )

    family_to_col = {
        family: i
        for i, family in enumerate(family_model.classes_)
    }

    # ---------------------------------------------
    # STAGE 2: PREDICT THE EXACT MODEL WITHIN EACH LAB
    # ---------------------------------------------

    for family in families:

        # Probability that each validation text belongs to this lab
        p_family = family_probs[
            :,
            family_to_col[family]
        ]

        # Select training rows belonging to this lab
        family_mask = np.array([
            MODEL_TO_FAMILY[model] == family
            for model in y_train
        ])

        X_family = X_train_features[family_mask]
        y_family = y_train[family_mask]

        family_labels = np.unique(y_family)

        # If the lab has only one candidate model,
        # its conditional probability is 1.
        if len(family_labels) == 1:

            model_label = family_labels[0]
            col = model_to_col[model_label]

            oof_probs[valid_idx, col] = p_family

        else:
            # Train a specialized classifier for this lab
            local_model = LogisticRegression(
                C=1.0,
                max_iter=2000,
            )

            local_model.fit(X_family, y_family)

            local_probs = local_model.predict_proba(
                X_valid_features
            )

            # Final probability:
            # P(exact model) = P(lab) * P(model | lab)
            for j, model_label in enumerate(
                local_model.classes_
            ):

                col = model_to_col[model_label]

                oof_probs[valid_idx, col] = (
                    p_family * local_probs[:, j]
                )

    # Evaluate this fold using all 18 model labels
    fold_predictions = all_models[
        np.argmax(oof_probs[valid_idx], axis=1)
    ]

    score = balanced_accuracy_score(
        y_valid,
        fold_predictions,
    )

    fold_scores.append(score)

    print(f"Fold balanced accuracy: {score:.4f}")


# =====================================================
# 5. DISPLAY FINAL CROSS-VALIDATION SCORE
# =====================================================

print("\n========== FINAL RESULTS ==========")

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(
    f"\nTwo-stage 5-fold Grouped CV Balanced Accuracy: "
    f"{np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}"
)

pooled_score = balanced_accuracy_score(
    y,
    all_models[np.argmax(oof_probs, axis=1)],
)

print(f"Pooled out-of-fold balanced accuracy: {pooled_score:.4f}")

print("\nPrevious best ensemble benchmark: 0.3589")


========== FOLD 1/5 ==========
Fold balanced accuracy: 0.2815

========== FOLD 2/5 ==========
Fold balanced accuracy: 0.3074

========== FOLD 3/5 ==========
Fold balanced accuracy: 0.3389

========== FOLD 4/5 ==========
Fold balanced accuracy: 0.3630

========== FOLD 5/5 ==========
Fold balanced accuracy: 0.3169

========== FINAL RESULTS ==========
Fold 1: 0.2815
Fold 2: 0.3074
Fold 3: 0.3389
Fold 4: 0.3630
Fold 5: 0.3169

Two-stage 5-fold Grouped CV Balanced Accuracy: 0.3215 +/- 0.0277
Pooled out-of-fold balanced accuracy: 0.3215

Previous best ensemble benchmark: 0.3589


In [26]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import GroupKFold
from sklearn.metrics import adjusted_rand_score

X = train_df["text"].fillna("")
y = train_df["model"]
groups = train_df["prompt_id"]

cv = GroupKFold(n_splits=5)

results = []

for fold, (_, valid_idx) in enumerate(
    cv.split(X, y, groups), start=1
):
    texts = X.iloc[valid_idx].tolist()
    true_prompts = groups.iloc[valid_idx].to_numpy()

    # Approximately 18 responses per prompt
    n_clusters = max(1, round(len(texts) / 18))

    # Represent passages using words and word pairs
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=40000,
        min_df=2,
        stop_words="english",
        sublinear_tf=True,
    )

    features = vectorizer.fit_transform(texts)

    # Calculate similarity between every pair of passages
    similarity = cosine_similarity(features)

    # Convert similarity into distance
    distance = np.clip(1.0 - similarity, 0.0, 2.0)
    np.fill_diagonal(distance, 0.0)

    # Cluster passages that appear to answer similar prompts
    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="precomputed",
        linkage="average",
    )

    predicted_clusters = clusterer.fit_predict(distance)

    # Diagnostic only: prompt IDs are NOT used in clustering
    ari = adjusted_rand_score(
        true_prompts,
        predicted_clusters,
    )

    results.append({
        "Fold": fold,
        "Number of clusters": n_clusters,
        "Adjusted Rand Index": ari,
    })

    print(
        f"Fold {fold}: "
        f"clusters={n_clusters}, ARI={ari:.4f}"
    )

results_df = pd.DataFrame(results)
display(results_df)

print(
    "Mean clustering ARI:",
    round(results_df["Adjusted Rand Index"].mean(), 4),
)

Fold 1: clusters=30, ARI=1.0000
Fold 2: clusters=30, ARI=1.0000
Fold 3: clusters=30, ARI=1.0000
Fold 4: clusters=30, ARI=1.0000
Fold 5: clusters=30, ARI=0.9573


,Fold,Number of clusters,Adjusted Rand Index
0,1,30,1.000000
1,2,30,1.000000
2,3,30,1.000000
3,4,30,1.000000
4,5,30,0.957254


Mean clustering ARI: 0.9915


In [27]:
import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold


# 1. Existing best ensemble
labels = np.sort(y.unique())

def geometric_blend(probabilities, weights):
    log_prob = np.zeros_like(
        probabilities[0], dtype=np.float64
    )

    for prob, weight in zip(probabilities, weights):
        log_prob += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_prob -= log_prob.max(axis=1, keepdims=True)
    combined = np.exp(log_prob)

    return combined / combined.sum(axis=1, keepdims=True)


oof_probs = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

baseline_predictions = labels[np.argmax(oof_probs, axis=1)]
constrained_predictions = baseline_predictions.copy()


# 2. Recover clusters separately inside each validation fold
cv = GroupKFold(n_splits=5)

fold_results = []
cluster_sizes_seen = []

for fold, (_, valid_idx) in enumerate(
    cv.split(X, y, groups), start=1
):
    texts = X.iloc[valid_idx].tolist()

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=40000,
        min_df=2,
        stop_words="english",
        sublinear_tf=True,
    )

    features = vectorizer.fit_transform(texts)

    similarity = (features @ features.T).toarray()
    distance = np.maximum(0.0, 1.0 - similarity)
    np.fill_diagonal(distance, 0.0)

    n_clusters = round(len(valid_idx) / 18)

    clusterer = AgglomerativeClustering(
        n_clusters=n_clusters,
        metric="precomputed",
        linkage="average",
    )

    cluster_ids = clusterer.fit_predict(distance)

    # 3. Assign model labels uniquely within valid clusters
    updated = 0

    for cluster_id in np.unique(cluster_ids):
        local_rows = np.where(cluster_ids == cluster_id)[0]
        global_rows = valid_idx[local_rows]

        size = len(global_rows)
        cluster_sizes_seen.append(size)

        # Training data has approximately 18 responses per prompt.
        # Test-set structure may differ, so skip unexpected sizes.
        if size not in (17, 18):
            continue

        log_probs = np.log(
            np.clip(oof_probs[global_rows], 1e-12, 1.0)
        )

        # If one response is missing, add a dummy row.
        # Its assigned label represents the missing model.
        if size == 17:
            log_probs = np.vstack([
                log_probs,
                np.zeros((1, len(labels))),
            ])

        # Find the assignment maximizing total log probability
        row_ind, col_ind = linear_sum_assignment(-log_probs)

        for r, c in zip(row_ind, col_ind):
            if r < size:
                constrained_predictions[global_rows[r]] = labels[c]

        updated += 1

    baseline_fold_score = balanced_accuracy_score(
        y.iloc[valid_idx],
        baseline_predictions[valid_idx],
    )

    constrained_fold_score = balanced_accuracy_score(
        y.iloc[valid_idx],
        constrained_predictions[valid_idx],
    )

    fold_results.append({
        "Fold": fold,
        "Baseline BA": baseline_fold_score,
        "Constrained BA": constrained_fold_score,
        "Clusters constrained": updated,
    })

    print(
        f"Fold {fold}: "
        f"baseline={baseline_fold_score:.4f}, "
        f"constrained={constrained_fold_score:.4f}, "
        f"clusters used={updated}/{n_clusters}"
    )


# 4. Compare results
results_df = pd.DataFrame(fold_results)
display(results_df)

print(
    "\nBaseline pooled BA:",
    round(
        balanced_accuracy_score(y, baseline_predictions), 4
    ),
)

print(
    "Constrained pooled BA:",
    round(
        balanced_accuracy_score(y, constrained_predictions), 4
    ),
)

print(
    "Observed cluster sizes:",
    pd.Series(cluster_sizes_seen).value_counts().sort_index().to_dict(),
)

Fold 1: baseline=0.3481, constrained=0.5037, clusters used=30/30
Fold 2: baseline=0.3222, constrained=0.4556, clusters used=30/30
Fold 3: baseline=0.4037, constrained=0.4944, clusters used=30/30
Fold 4: baseline=0.3815, constrained=0.5019, clusters used=30/30
Fold 5: baseline=0.3391, constrained=0.4542, clusters used=27/30


,Fold,Baseline BA,Constrained BA,Clusters constrained
0,1,0.348148,0.503704,30
1,2,0.322222,0.455556,30
2,3,0.403704,0.494444,30
3,4,0.381481,0.501852,30
4,5,0.339080,0.454215,27



Baseline pooled BA: 0.3589
Constrained pooled BA: 0.482
Observed cluster sizes: {5: 1, 13: 1, 17: 1, 18: 146, 36: 1}


In [2]:
# ============================================================
# 1. IMPORTS AND DATA
# ============================================================

import pathlib
import warnings
import numpy as np
import pandas as pd
import torch
import transformers

from packaging.version import Version
from tqdm.auto import tqdm

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)

# Load data if not already loaded
if "train_df" not in globals() or "test_df" not in globals():

    CANDIDATES = [
        pathlib.Path("/kaggle/input/competitions/name-that-llm"),
        pathlib.Path("/kaggle/input/name-that-llm"),
    ]

    DATA_DIR = next(
        (p for p in CANDIDATES if (p / "train.csv").exists()),
        None,
    )

    if DATA_DIR is None:
        raise FileNotFoundError(
            "Could not find train.csv. Attach the competition dataset."
        )

    train_df = pd.read_csv(DATA_DIR / "train.csv")
    test_df = pd.read_csv(DATA_DIR / "test.csv")

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

labels = np.sort(y.unique())

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))

print("Training rows:", len(X))
print("Test rows:", len(test_df))
print("Classes:", len(labels))


# ============================================================
# 2. LOCATE OR DOWNLOAD QWEN3 EMBEDDING
# ============================================================

from pathlib import Path

model_dir = None

# First, look through attached Kaggle model inputs.
for config_file in Path("/kaggle/input").rglob("config.json"):

    candidate = config_file.parent
    path_text = str(candidate).lower()

    is_qwen_embedding = (
        "qwen" in path_text
        and "embedding" in path_text
        and any(token in path_text for token in [
            "0.6b", "0-6b", "0_6b", "06b"
        ])
    )

    if is_qwen_embedding:
        model_dir = candidate
        break

# If it is not attached, download it using KaggleHub.
if model_dir is None:
    try:
        import kagglehub

        model_dir = Path(
            kagglehub.model_download(
                "qwen-lm/qwen-3-embedding/transformers/0.6b"
            )
        )

    except Exception as exc:
        raise RuntimeError(
            "Could not find or download Qwen3 Embedding 0.6B.\n"
            "Enable Internet in Kaggle settings, or attach the model "
            "using Add Input -> Models -> Qwen 3 Embedding -> Transformers 0.6b."
        ) from exc

if not (Path(model_dir) / "config.json").exists():
    raise FileNotFoundError(
        f"Model directory has no config.json: {model_dir}"
    )

print("Model directory:", model_dir)


# ============================================================
# 3. LOAD THE PRETRAINED MODEL
# ============================================================

from transformers import AutoTokenizer, AutoModel

if Version(transformers.__version__) < Version("4.51.0"):
    raise RuntimeError(
        f"Transformers version {transformers.__version__} is too old. "
        "Qwen3 requires transformers >= 4.51.0."
    )

device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    dtype = torch.float16
    batch_size = 4
else:
    dtype = torch.float32
    batch_size = 2
    print("Warning: CPU inference may be slow.")

tokenizer = AutoTokenizer.from_pretrained(
    str(model_dir),
    local_files_only=True,
    padding_side="left",
)

tokenizer.padding_side = "left"

model = AutoModel.from_pretrained(
    str(model_dir),
    local_files_only=True,
    torch_dtype=dtype,
)

model.to(device)
model.eval()

print("Device:", device)
print("Model loaded successfully.")


# ============================================================
# 4. CONVERT TEXT INTO EMBEDDINGS
# ============================================================

def encode_texts(texts, batch_size=4, max_length=1024):

    embeddings_list = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Creating embeddings",
    ):

        batch_texts = texts[start:start + batch_size]

        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        tokens = {
            key: value.to(device)
            for key, value in tokens.items()
        }

        with torch.inference_mode():

            outputs = model(**tokens)

            # Left padding means the final position is the
            # final actual token for each passage.
            batch_embeddings = outputs.last_hidden_state[:, -1, :]

            # Normalize each passage embedding.
            batch_embeddings = torch.nn.functional.normalize(
                batch_embeddings,
                p=2,
                dim=1,
            )

        embeddings_list.append(
            batch_embeddings.float().cpu().numpy()
        )

    if not embeddings_list:
        return np.empty((0, model.config.hidden_size), dtype=np.float32)

    return np.concatenate(embeddings_list, axis=0)


all_texts = X.tolist() + test_df["text"].fillna("").astype(str).tolist()

all_embeddings = encode_texts(
    all_texts,
    batch_size=batch_size,
    max_length=1024,
)

X_embed = all_embeddings[:len(X)]
X_test_embed = all_embeddings[len(X):]

del all_embeddings

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Training embeddings:", X_embed.shape)
print("Test embeddings:", X_test_embed.shape)


# ============================================================
# 5. BASELINE MODELS — REUSE OOF OR BUILD THEM
# ============================================================

def build_word_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=25000,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=1500,
        )),
    ])


def build_char_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(1, 5),
            max_features=100000,
            sublinear_tf=True,
        )),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=1500,
        )),
    ])


class StyleFeatureExtractor(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "as an ai", "as a language model", "certainly",
        "of course", "here is", "here's", "let's",
        "in conclusion", "in summary", "to summarize",
        "for example", "for instance", "note that",
        "it is important", "it's important", "overall",
        "however", "therefore", "moreover", "furthermore",
        "firstly", "secondly", "hope this helps",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract_features(text) for text in X],
            dtype=np.float32,
        )

    def extract_features(self, value):

        text = "" if value is None else str(value)
        lower = text.lower()

        char_count = max(len(text), 1)
        words = __import__("re").findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)

        word_lengths = np.asarray(
            [len(w) for w in words],
            dtype=np.float32,
        )

        import re

        sentences = [
            s.strip()
            for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = np.asarray(
            [
                len(re.findall(r"\b\w+\b", s))
                for s in sentences
            ],
            dtype=np.float32,
        )

        lines = text.splitlines() or [text]

        paragraphs = [
            p for p in re.split(r"\n\s*\n", text)
            if p.strip()
        ]

        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),
            float(word_lengths.mean()) if word_lengths.size else 0.0,
            float(word_lengths.std()) if word_lengths.size else 0.0,
            float(word_lengths.max()) if word_lengths.size else 0.0,
            len(set(words)) / word_count,
            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,
            float(sentence_lengths.mean()) if sentence_lengths.size else 0.0,
            float(sentence_lengths.std()) if sentence_lengths.size else 0.0,
            float(sentence_lengths.max()) if sentence_lengths.size else 0.0,
        ]

        features.extend([
            sum(c.isupper() for c in text) / char_count,
            sum(c.islower() for c in text) / char_count,
            sum(c.isdigit() for c in text) / char_count,
            sum(c.isspace() for c in text) / char_count,
            sum(ord(c) > 127 for c in text) / char_count,
        ])

        for mark in self.punctuation:
            features.append(text.count(mark) / char_count * 1000)

        bullet_lines = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )

        features.extend([
            bullet_lines / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        for phrase in self.phrases:
            features.append(float(phrase in lower))

        return features


def build_style_model():
    return Pipeline([
        ("style_features", StyleFeatureExtractor()),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=1500,
        )),
    ])


have_baseline_oof = all(
    name in globals()
    for name in ["word_oof", "char_oof", "style_oof"]
)

if have_baseline_oof:
    have_baseline_oof = all(
        np.asarray(globals()[name]).shape == (len(X), len(labels))
        for name in ["word_oof", "char_oof", "style_oof"]
    )

if not have_baseline_oof:

    print("Existing baseline OOF predictions not found. Rebuilding...")

    print("Generating word-model OOF predictions...")
    word_oof = cross_val_predict(
        build_word_model(), X, y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

    print("Generating character-model OOF predictions...")
    char_oof = cross_val_predict(
        build_char_model(), X, y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

    print("Generating style-model OOF predictions...")
    style_oof = cross_val_predict(
        build_style_model(), X, y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

print("Baseline OOF probabilities are ready.")


# ============================================================
# 6. EVALUATE EMBEDDING CLASSIFIER
# ============================================================

embedding_results = []

best_embedding_score = -1
best_embedding_c = None
best_embedding_oof = None

for c in [0.03, 0.1, 0.3, 1.0, 3.0]:

    print(f"\nEvaluating embedding Logistic Regression C={c}")

    classifier = LogisticRegression(
        C=c,
        max_iter=2000,
    )

    oof_probs = cross_val_predict(
        classifier,
        X_embed,
        y,
        cv=cv_splits,
        method="predict_proba",
        n_jobs=1,
    )

    predictions = labels[np.argmax(oof_probs, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    embedding_results.append({
        "C": c,
        "Balanced accuracy": score,
    })

    print(f"Balanced accuracy: {score:.4f}")

    if score > best_embedding_score:
        best_embedding_score = score
        best_embedding_c = c
        best_embedding_oof = oof_probs.copy()


embedding_results_df = pd.DataFrame(embedding_results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== EMBEDDING RESULTS ==========")
display(embedding_results_df)

print("Best C:", best_embedding_c)
print("Best embedding score:", round(best_embedding_score, 4))


# ============================================================
# 7. COMPARE AGAINST THE EXISTING ENSEMBLE
# ============================================================

def geometric_blend(probabilities, weights):

    log_probs = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for probs, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(probs, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)

    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


def rank_scores(probabilities):

    from scipy.stats import rankdata

    ranks = rankdata(
        probabilities,
        axis=1,
        method="average",
    )

    return (ranks - 1) / (probabilities.shape[1] - 1)


baseline_oof = geometric_blend(
    [word_oof, char_oof, style_oof],
    [0.40, 0.50, 0.10],
)

baseline_score = balanced_accuracy_score(
    y,
    labels[np.argmax(baseline_oof, axis=1)],
)

print(f"\nExisting ensemble score: {baseline_score:.4f}")

baseline_ranks = rank_scores(baseline_oof)
embedding_ranks = rank_scores(best_embedding_oof)

blend_results = [{
    "Method": "Existing ensemble",
    "Embedding weight": 0.0,
    "Balanced accuracy": baseline_score,
}]

blend_results.append({
    "Method": "Embedding alone",
    "Embedding weight": 1.0,
    "Balanced accuracy": best_embedding_score,
})

for embedding_weight in [0.20, 0.35, 0.50, 0.65]:

    combined = (
        (1 - embedding_weight) * baseline_ranks
        + embedding_weight * embedding_ranks
    )

    predictions = labels[np.argmax(combined, axis=1)]

    score = balanced_accuracy_score(y, predictions)

    blend_results.append({
        "Method": "Rank blend",
        "Embedding weight": embedding_weight,
        "Balanced accuracy": score,
    })


blend_results_df = pd.DataFrame(blend_results).sort_values(
    "Balanced accuracy",
    ascending=False,
).reset_index(drop=True)

print("\n========== BLEND COMPARISON ==========")
display(blend_results_df)

best_embedding_weight = float(
    blend_results_df.iloc[0]["Embedding weight"]
)

print("Selected embedding weight:", best_embedding_weight)


# ============================================================
# 8. FIT ALL MODELS ON THE COMPLETE TRAINING DATA
# ============================================================

word_model = build_word_model()
char_model = build_char_model()
style_model = build_style_model()

print("\nTraining final baseline models...")

word_model.fit(X, y)
char_model.fit(X, y)
style_model.fit(X, y)

print("Training final embedding classifier...")

embedding_model = LogisticRegression(
    C=best_embedding_c,
    max_iter=2000,
)

embedding_model.fit(X_embed, y)


# ============================================================
# 9. GENERATE TEST PREDICTIONS
# ============================================================

test_texts = test_df["text"].fillna("").astype(str)

word_test = word_model.predict_proba(test_texts)
char_test = char_model.predict_proba(test_texts)
style_test = style_model.predict_proba(test_texts)

baseline_test = geometric_blend(
    [word_test, char_test, style_test],
    [0.40, 0.50, 0.10],
)

embedding_test = embedding_model.predict_proba(X_test_embed)

baseline_test_ranks = rank_scores(baseline_test)
embedding_test_ranks = rank_scores(embedding_test)

final_scores = (
    (1 - best_embedding_weight) * baseline_test_ranks
    + best_embedding_weight * embedding_test_ranks
)

final_predictions = labels[np.argmax(final_scores, axis=1)]


# ============================================================
# 10. SAVE SUBMISSION
# ============================================================

submission = pd.DataFrame({
    "id": test_df["id"],
    "model": final_predictions,
})

submission.to_parquet(
    "/kaggle/working/submission.parquet",
    index=False,
)

print("\nSubmission saved successfully.")
print("Rows:", len(submission))
display(submission.head(10))

Training rows: 2699
Test rows: 3
Classes: 18
Model directory: /kaggle/input/models/qwen-lm/qwen-3-embedding/transformers/0.6b/1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Device: cuda
Model loaded successfully.


Creating embeddings:   0%|          | 0/676 [00:00<?, ?it/s]

...

,C,Balanced accuracy
0,3.00,0.167427
1,1.00,0.164822
2,0.30,0.154074
3,0.10,0.141111
4,0.03,0.135926


Best C: 3.0
Best embedding score: 0.1674

Existing ensemble score: 0.3600

========== BLEND COMPARISON ==========


,Method,Embedding weight,Balanced accuracy
0,Existing ensemble,0.00,0.360032
1,Rank blend,0.20,0.354119
2,Rank blend,0.35,0.323005
3,Rank blend,0.50,0.299302
4,Rank blend,0.65,0.263731
5,Embedding alone,1.00,0.167427


Selected embedding weight: 0.0

Training final baseline models...
Training final embedding classifier...

Submission saved successfully.
Rows: 3


,id,model
0,00353fb1c541,gemini-3.6-flash
1,005f5a9a6139,gpt-5.6-sol
2,0062a53d5487,gemini-3.7-flash


In [3]:
# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import pandas as pd
import random
import torch

from pathlib import Path
from torch.utils.data import TensorDataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score

from scipy.stats import rankdata


# ============================================================
# 2. CONFIGURATION
# ============================================================

MODEL_NAME = "google-bert/bert-base-cased"

MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)

if DEVICE.type != "cuda":
    print(
        "WARNING: BERT fine-tuning on CPU will be slow. "
        "Enable a GPU in Kaggle settings."
    )


# ============================================================
# 3. PREPARE DATA
# ============================================================

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].astype(str).reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

labels = label_encoder.classes_
NUM_LABELS = len(labels)

print("Training examples:", len(X))
print("Number of classes:", NUM_LABELS)

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y_encoded, groups))

# Tokenizer does not use labels, so we can tokenize once.
# BERT will process at most 256 tokens per passage in this run.

# If the model is attached as a Kaggle input, use its local path.
local_model_paths = [
    p.parent
    for p in Path("/kaggle/input").rglob("config.json")
    if "bert-base-cased" in str(p.parent).lower()
]

if local_model_paths:
    MODEL_NAME = str(local_model_paths[0])

print("Model source:", MODEL_NAME)


# ============================================================
# 4. LOAD TOKENIZER AND TOKENIZE TEXT
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    local_files_only=Path(MODEL_NAME).exists(),
)

encoded = tokenizer(
    X.tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

input_ids = encoded["input_ids"]
attention_masks = encoded["attention_mask"]

print("Token tensor shape:", input_ids.shape)


# ============================================================
# 5. FUNCTION TO CREATE A FRESH BERT CLASSIFIER
# ============================================================

def build_bert():

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={
            i: str(label)
            for i, label in enumerate(labels)
        },
        label2id={
            str(label): i
            for i, label in enumerate(labels)
        },
    )

    return model.to(DEVICE)


# ============================================================
# 6. FIVE-FOLD GROUPED CROSS-VALIDATION
# ============================================================

bert_oof_probs = np.zeros(
    (len(X), NUM_LABELS),
    dtype=np.float32,
)

fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(
    cv_splits, start=1
):

    print(f"\n{'=' * 45}")
    print(f"FOLD {fold}/5")
    print(f"{'=' * 45}")

    # Reproducibility
    seed = 42 + fold
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # --------------------------------------------------------
    # Create the fold-specific datasets
    # --------------------------------------------------------

    train_indices = torch.tensor(
        train_idx, dtype=torch.long
    )
    valid_indices = torch.tensor(
        valid_idx, dtype=torch.long
    )

    train_dataset = TensorDataset(
        input_ids[train_indices],
        attention_masks[train_indices],
        torch.tensor(
            y_encoded[train_idx],
            dtype=torch.long,
        ),
    )

    valid_dataset = TensorDataset(
        input_ids[valid_indices],
        attention_masks[valid_indices],
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=USE_AMP,
    )

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=32,
        shuffle=False,
        pin_memory=USE_AMP,
    )

    # --------------------------------------------------------
    # Load pretrained BERT and initialize classification head
    # --------------------------------------------------------

    model = build_bert()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(total_steps * 0.06),
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )

    # --------------------------------------------------------
    # Fine-tune BERT
    # --------------------------------------------------------

    for epoch in range(EPOCHS):

        model.train()
        total_loss = 0.0

        for batch in train_loader:

            batch_input_ids = batch[0].to(DEVICE)
            batch_attention = batch[1].to(DEVICE)
            batch_labels = batch[2].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=batch_input_ids,
                    attention_mask=batch_attention,
                    labels=batch_labels,
                )

                loss = outputs.loss

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        mean_loss = total_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{EPOCHS} "
            f"| Training loss: {mean_loss:.4f}"
        )

    # --------------------------------------------------------
    # Predict the validation fold
    # --------------------------------------------------------

    model.eval()
    fold_probabilities = []

    with torch.no_grad():

        for batch in valid_loader:

            batch_input_ids = batch[0].to(DEVICE)
            batch_attention = batch[1].to(DEVICE)

            with torch.cuda.amp.autocast(
                enabled=USE_AMP
            ):

                outputs = model(
                    input_ids=batch_input_ids,
                    attention_mask=batch_attention,
                )

            probabilities = torch.softmax(
                outputs.logits.float(),
                dim=-1,
            )

            fold_probabilities.append(
                probabilities.cpu().numpy()
            )

    fold_probabilities = np.concatenate(
        fold_probabilities,
        axis=0,
    )

    bert_oof_probs[valid_idx] = fold_probabilities

    fold_predictions = labels[
        np.argmax(fold_probabilities, axis=1)
    ]

    score = balanced_accuracy_score(
        y.iloc[valid_idx],
        fold_predictions,
    )

    fold_scores.append(score)

    print(f"Fold balanced accuracy: {score:.4f}")

    del model, optimizer, scheduler, scaler
    del train_loader, valid_loader
    del train_dataset, valid_dataset

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# 7. REPORT BERT PERFORMANCE
# ============================================================

print("\n========== BERT CROSS-VALIDATION RESULTS ==========")

for i, score in enumerate(fold_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print(
    f"\nMean fold balanced accuracy: "
    f"{np.mean(fold_scores):.4f} "
    f"+/- {np.std(fold_scores):.4f}"
)

bert_oof_predictions = labels[
    np.argmax(bert_oof_probs, axis=1)
]

bert_pooled_score = balanced_accuracy_score(
    y,
    bert_oof_predictions,
)

print(
    f"Pooled out-of-fold balanced accuracy: "
    f"{bert_pooled_score:.4f}"
)

print("Previous best ensemble: 0.3589")


# ============================================================
# 8. OPTIONAL: COMPARE BERT WITH EXISTING ENSEMBLE
# ============================================================

def geometric_blend(probabilities, weights):

    log_probs = np.zeros_like(
        probabilities[0],
        dtype=np.float64,
    )

    for prob, weight in zip(probabilities, weights):
        log_probs += weight * np.log(
            np.clip(prob, 1e-12, 1.0)
        )

    log_probs -= log_probs.max(axis=1, keepdims=True)

    combined = np.exp(log_probs)

    return combined / combined.sum(axis=1, keepdims=True)


def rank_scores(probabilities):

    ranks = rankdata(
        probabilities,
        axis=1,
        method="average",
    )

    return (ranks - 1) / (probabilities.shape[1] - 1)


if all(
    name in globals()
    for name in ["word_oof", "char_oof", "style_oof"]
):

    old_ensemble = geometric_blend(
        [word_oof, char_oof, style_oof],
        [0.40, 0.50, 0.10],
    )

    old_score = balanced_accuracy_score(
        y,
        labels[np.argmax(old_ensemble, axis=1)],
    )

    blend_results = [
        {
            "Method": "Existing ensemble",
            "BERT weight": 0.0,
            "Balanced accuracy": old_score,
        },
        {
            "Method": "BERT alone",
            "BERT weight": 1.0,
            "Balanced accuracy": bert_pooled_score,
        },
    ]

    old_ranks = rank_scores(old_ensemble)
    bert_ranks = rank_scores(bert_oof_probs)

    for bert_weight in [0.10, 0.20, 0.30, 0.40]:

        combined_ranks = (
            (1 - bert_weight) * old_ranks
            + bert_weight * bert_ranks
        )

        predictions = labels[
            np.argmax(combined_ranks, axis=1)
        ]

        score = balanced_accuracy_score(y, predictions)

        blend_results.append({
            "Method": "Rank blend",
            "BERT weight": bert_weight,
            "Balanced accuracy": score,
        })

    blend_results_df = pd.DataFrame(
        blend_results
    ).sort_values(
        "Balanced accuracy",
        ascending=False,
    ).reset_index(drop=True)

    print("\n========== ENSEMBLE COMPARISON ==========")
    display(blend_results_df)

else:

    print(
        "\nExisting ensemble OOF arrays were not found. "
        "BERT was evaluated independently."
    )

Device: cuda
Training examples: 2699
Number of classes: 18
Model source: google-bert/bert-base-cased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykern

Epoch 1/3 | Training loss: 2.7627
Epoch 2/3 | Training loss: 2.4719
Epoch 3/3 | Training loss: 2.2872


/tmp/ipykernel_49/2215965741.py:288: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Fold balanced accuracy: 0.2169

========== BERT CROSS-VALIDATION RESULTS ==========
Fold 1: 0.2537
Fold 2: 0.1426
Fold 3: 0.2333
Fold 4: 0.1685
Fold 5: 0.2169

Mean fold balanced accuracy: 0.2030 +/- 0.0413
Pooled out-of-fold balanced accuracy: 0.2030
Previous best ensemble: 0.3589

Existing ensemble OOF arrays were not found. BERT was evaluated independently.


In [4]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.base import BaseEstimator, TransformerMixin


# --------------------------------------------------
# 1. Prepare data and fixed validation folds
# --------------------------------------------------

X = train_df["text"].fillna("").astype(str).reset_index(drop=True)
y = train_df["model"].reset_index(drop=True)
groups = train_df["prompt_id"].reset_index(drop=True)

cv = GroupKFold(n_splits=5)
cv_splits = list(cv.split(X, y, groups))


# --------------------------------------------------
# 2. Extract stylistic features
# --------------------------------------------------

class StyleFeatures(BaseEstimator, TransformerMixin):

    punctuation = [
        ",", ";", ":", ".", "!", "?", "-", "—", "–",
        "'", "’", '"', "“", "”", "(", ")", "[", "]",
        "{", "}", "/", "\\", "*", "_", "#", "`", "…", "•"
    ]

    phrases = [
        "certainly", "of course", "for example",
        "for instance", "in summary", "in conclusion",
        "however", "therefore", "moreover",
        "furthermore", "it is important", "overall",
        "hope this helps", "let's", "here is",
        "as a language model", "note that",
    ]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.asarray(
            [self.extract(text) for text in X],
            dtype=np.float32,
        )

    def extract(self, text):
        text = str(text)
        lower = text.lower()
        length = max(len(text), 1)

        words = re.findall(r"\b\w+\b", lower)
        word_count = max(len(words), 1)

        word_lengths = [len(w) for w in words]

        sentences = [
            s.strip() for s in re.split(r"[.!?]+", text)
            if s.strip()
        ]

        sentence_lengths = [
            len(re.findall(r"\b\w+\b", s))
            for s in sentences
        ]

        lines = text.splitlines() or [text]
        paragraphs = [
            p for p in re.split(r"\n\s*\n", text) if p.strip()
        ]

        features = [
            np.log1p(len(text)),
            np.log1p(len(words)),
            np.log1p(len(sentences)),
            np.log1p(len(lines)),
            np.log1p(len(paragraphs)),
            np.mean(word_lengths) if word_lengths else 0,
            np.std(word_lengths) if word_lengths else 0,
            np.max(word_lengths) if word_lengths else 0,
            len(set(words)) / word_count,
            sum(len(w) <= 3 for w in words) / word_count,
            sum(len(w) >= 8 for w in words) / word_count,
            np.mean(sentence_lengths) if sentence_lengths else 0,
            np.std(sentence_lengths) if sentence_lengths else 0,
            np.max(sentence_lengths) if sentence_lengths else 0,
            sum(c.isupper() for c in text) / length,
            sum(c.islower() for c in text) / length,
            sum(c.isdigit() for c in text) / length,
            sum(c.isspace() for c in text) / length,
            sum(ord(c) > 127 for c in text) / length,
        ]

        # Punctuation frequencies
        features.extend(
            text.count(mark) * 1000 / length
            for mark in self.punctuation
        )

        # Formatting indicators
        bullet_count = sum(
            bool(re.match(r"^\s*(?:[-*•]\s+|\d+[.)]\s+)", line))
            for line in lines
        )

        features.extend([
            bullet_count / max(len(lines), 1),
            float("\n" in text),
            float("```" in text),
            float("##" in text),
            float(text.rstrip().endswith(("!", "?", "."))),
        ])

        features.extend(
            float(phrase in lower)
            for phrase in self.phrases
        )

        return features


# --------------------------------------------------
# 3. Build multi-view SVM
# --------------------------------------------------

def build_model(C, style_weight):

    return Pipeline([
        ("features", FeatureUnion(
            transformer_list=[
                ("word", TfidfVectorizer(
                    ngram_range=(1, 2),
                    max_features=25000,
                    sublinear_tf=True,
                )),

                ("char", TfidfVectorizer(
                    analyzer="char",
                    ngram_range=(1, 5),
                    max_features=100000,
                    sublinear_tf=True,
                )),

                ("style", Pipeline([
                    ("extract", StyleFeatures()),
                    ("scale", StandardScaler()),
                ])),
            ],
            transformer_weights={
                "word": 1.0,
                "char": 1.0,
                "style": style_weight,
            },
        )),
        ("classifier", LinearSVC(C=C)),
    ])


# --------------------------------------------------
# 4. Compare nine configurations
# --------------------------------------------------

results = []

for C in [0.5, 1.5, 3.0]:
    for style_weight in [0.1, 0.2, 0.3]:

        print(f"\nTesting C={C}, style_weight={style_weight}")

        model = build_model(C, style_weight)

        scores = cross_val_score(
            model,
            X,
            y,
            groups=groups,
            cv=cv_splits,
            scoring="balanced_accuracy",
            n_jobs=-1,
        )

        results.append({
            "C": C,
            "Style weight": style_weight,
            "Mean balanced accuracy": scores.mean(),
            "Std deviation": scores.std(),
            "Fold scores": scores.round(4).tolist(),
        })

        print(f"Fold scores: {scores.round(4)}")
        print(f"Mean: {scores.mean():.4f} +/- {scores.std():.4f}")


# --------------------------------------------------
# 5. Display the best configuration
# --------------------------------------------------

results_df = pd.DataFrame(results).sort_values(
    "Mean balanced accuracy",
    ascending=False,
).reset_index(drop=True)

display(results_df)

print("\nBest configuration:")
display(results_df.head(1))

print("\nCurrent ensemble benchmark: 0.3589")


Testing C=0.5, style_weight=0.1
Fold scores: [0.363  0.3426 0.3852 0.3704 0.3465]
Mean: 0.3615 +/- 0.0157

Testing C=0.5, style_weight=0.2
Fold scores: [0.3556 0.3444 0.3815 0.3593 0.3446]
Mean: 0.3571 +/- 0.0135

Testing C=0.5, style_weight=0.3
Fold scores: [0.3611 0.3407 0.3741 0.35   0.3372]
Mean: 0.3526 +/- 0.0135

Testing C=1.5, style_weight=0.1
Fold scores: [0.3778 0.3556 0.3815 0.3685 0.3466]
Mean: 0.3660 +/- 0.0132

Testing C=1.5, style_weight=0.2
Fold scores: [0.3759 0.3426 0.3722 0.3611 0.3372]
Mean: 0.3578 +/- 0.0155

Testing C=1.5, style_weight=0.3


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold scores: [0.3722 0.3444 0.3741 0.3648 0.3372]
Mean: 0.3586 +/- 0.0150

Testing C=3.0, style_weight=0.1
Fold scores: [0.3759 0.3519 0.3778 0.3667 0.3447]
Mean: 0.3634 +/- 0.0131

Testing C=3.0, style_weight=0.2
Fold scores: [0.3759 0.3463 0.3741 0.3593 0.341 ]
Mean: 0.3593 +/- 0.0141

Testing C=3.0, style_weight=0.3


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Fold scores: [0.3759 0.3426 0.3741 0.3667 0.3317]
Mean: 0.3582 +/- 0.0178


,C,Style weight,Mean balanced accuracy,Std deviation,Fold scores
0,1.5,0.1,0.365977,0.013206,"[0.3778, 0.3556, 0.3815, 0.3685, 0.3466]"
1,3.0,0.1,0.363384,0.013095,"[0.3759, 0.3519, 0.3778, 0.3667, 0.3447]"
2,0.5,0.1,0.361520,0.015650,"[0.363, 0.3426, 0.3852, 0.3704, 0.3465]"
3,3.0,0.2,0.359310,0.014134,"[0.3759, 0.3463, 0.3741, 0.3593, 0.341]"
4,1.5,0.3,0.358557,0.014972,"[0.3722, 0.3444, 0.3741, 0.3648, 0.3372]"
5,3.0,0.3,0.358186,0.017805,"[0.3759, 0.3426, 0.3741, 0.3667, 0.3317]"
6,1.5,0.2,0.357816,0.015505,"[0.3759, 0.3426, 0.3722, 0.3611, 0.3372]"
7,0.5,0.2,0.357075,0.013543,"[0.3556, 0.3444, 0.3815, 0.3593, 0.3446]"
8,0.5,0.3,0.352631,0.013544,"[0.3611, 0.3407, 0.3741, 0.35, 0.3372]"



Best configuration:


,C,Style weight,Mean balanced accuracy,Std deviation,Fold scores
0,1.5,0.1,0.365977,0.013206,"[0.3778, 0.3556, 0.3815, 0.3685, 0.3466]"



Current ensemble benchmark: 0.3589
